In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# LangChain

In [3]:
import os
import asyncio
from typing import List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain.agents import create_agent

# --- Configuration ---
# Ensure your GOOGLE_API_KEY environment variable is set.
try:
    # A model with function/tool calling capabilities is required.
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    print(f"Language model initialized: {llm.model}")
except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None


# --- Define a Tool ---
@tool
def search_information(query: str) -> str:
    """
    Provides factual information on a given topic. Use this tool to find answers to questions
    like 'What is the capital of France?' or 'What is the weather in London?'.
    """
    print(f"\n--- Tool Called: search_information with query: '{query}' ---")
    # Simulate a search tool with a dictionary of predefined results.
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
        "default": f"Simulated search result for '{query}': No specific information found, but the topic seems interesting."
    }
    result = simulated_results.get(query.lower(), simulated_results["default"])
    print(f"--- TOOL RESULT: {result} ---")
    return result

tools = [search_information]


# --- Create a Tool-Calling Agent ---
if llm:
    # create_agent builds the agent loop: the model decides which tool to call, the
    # tool runs, its result is fed back to the model, and the loop repeats until the
    # model answers. No separate executor and no agent_scratchpad placeholder needed.
    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="You are a helpful assistant.",
    )


    async def run_agent_with_tool(query: str):
        """Invokes the agent with a query and prints the final response."""
        print(f"\n--- Running Agent with Query: '{query}' ---")
        try:
            response = await agent.ainvoke(
                {"messages": [{"role": "user", "content": query}]}
            )
            print("\n--- Final Agent Response ---")
            # The agent returns the whole message history; the last message is the answer.
            print(response["messages"][-1].text)
        except Exception as e:
            print(f"\nAn error occurred during agent execution: {e}")

    async def main():
        """Runs all agent queries concurrently."""
        tasks = [
            run_agent_with_tool("What is the capital of France?"),
            run_agent_with_tool("What's the weather like in London?"),
            run_agent_with_tool("Tell me something about dogs.") # Should trigger the default tool response
        ]
        await asyncio.gather(*tasks)

    # The notebook already runs an event loop, so await the coroutine directly.
    await main()

else:
    print("\nSkipping agent execution due to LLM initialization failure.")


Language model initialized: gemini-2.5-flash

--- Running Agent with Query: 'What is the capital of France?' ---

--- Running Agent with Query: 'What's the weather like in London?' ---

--- Running Agent with Query: 'Tell me something about dogs.' ---

--- Tool Called: search_information with query: 'dogs' ---
--- Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: Simulated search result for 'What is the capital of France?': No specific information found, but the topic seems interesting. ---

--- TOOL RESULT: Simulated search result for 'dogs': No specific information found, but the topic seems interesting. ---

--- Tool Called: search_information with query: 'weather in London' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---

--- Final Agent Response ---
The weather in London is currently cloudy with a temperature of 15°C.

--- Final Agent Response ---
I don't have information about the capit

# CrewAI

In [4]:
# pip install crewai langchain-openai

import os
from crewai import Agent, Task, Crew
from crewai.tools import tool
import logging

# --- Best Practice: Configure Logging ---
# A basic logging setup helps in debugging and tracking the crew's execution.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Set up your API Key ---
# For production, it's recommended to use a more secure method for key management
# like environment variables loaded at runtime or a secret manager.
#
# Set the environment variable for your chosen LLM provider (e.g., OPENAI_API_KEY)
# os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
# os.environ["OPENAI_MODEL_NAME"] = "gpt-4o"


# --- 1. Refactored Tool: Returns Clean Data ---
# The tool now returns raw data (a float) or raises a standard Python error.
# This makes it more reusable and forces the agent to handle outcomes properly.
@tool("Stock Price Lookup Tool")
def get_stock_price(ticker: str) -> float:
    """
    Fetches the latest simulated stock price for a given stock ticker symbol.
    Returns the price as a float. Raises a ValueError if the ticker is not found.
    """
    logging.info(f"Tool Call: get_stock_price for ticker '{ticker}'")
    simulated_prices = {
        "AAPL": 178.15,
        "GOOGL": 1750.30,
        "MSFT": 425.50,
    }
    price = simulated_prices.get(ticker.upper())

    if price is not None:
        return price
    else:
        # Raising a specific error is better than returning a string.
        # The agent is equipped to handle exceptions and can decide on the next action.
        raise ValueError(f"Simulated price for ticker '{ticker.upper()}' not found.")


# --- 2. Define the Agent ---
# The agent definition remains the same, but it will now leverage the improved tool.
financial_analyst_agent = Agent(
  role='Senior Financial Analyst',
  goal='Analyze stock data using provided tools and report key prices.',
  backstory="You are an experienced financial analyst adept at using data sources to find stock information. You provide clear, direct answers.",
  verbose=True,
  tools=[get_stock_price],
  # Allowing delegation can be useful, but is not necessary for this simple task.
  allow_delegation=False,
)

# --- 3. Refined Task: Clearer Instructions and Error Handling ---
# The task description is more specific and guides the agent on how to react
# to both successful data retrieval and potential errors.
analyze_aapl_task = Task(
  description=(
      "What is the current simulated stock price for Apple (ticker: AAPL)? "
      "Use the 'Stock Price Lookup Tool' to find it. "
      "If the ticker is not found, you must report that you were unable to retrieve the price."
  ),
  expected_output=(
      "A single, clear sentence stating the simulated stock price for AAPL. "
      "For example: 'The simulated stock price for AAPL is $178.15.' "
      "If the price cannot be found, state that clearly."
  ),
  agent=financial_analyst_agent,
)

# --- 4. Formulate the Crew ---
# The crew orchestrates how the agent and task work together.
financial_crew = Crew(
  agents=[financial_analyst_agent],
  tasks=[analyze_aapl_task],
  verbose=True # Set to False for less detailed logs in production
)

# --- 5. Run the Crew ---
async def main():
    """Main function to run the crew."""
    # Check for API key before starting to avoid runtime errors.
    if not os.environ.get("OPENAI_API_KEY"):
        print("ERROR: The OPENAI_API_KEY environment variable is not set.")
        print("Please set it before running the script.")
        return

    print("\n## Starting the Financial Crew...")
    print("---------------------------------")

    # The notebook already runs an event loop, so use the async kickoff.
    # The synchronous kickoff() raises RuntimeError when called from one.
    result = await financial_crew.kickoff_async()

    print("\n---------------------------------")
    print("## Crew execution finished.")
    print("\nFinal Result:\n", result)

await main()


## Starting the Financial Crew...
---------------------------------


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 86344487-8bf8-4652-bd15-305077e9da35                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│  ID: bf716aa8-f753-48b8-9f28-0259ff0a291b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│  Task: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-08-25 23:01:49,447 - INFO - OpenAI: Successfully validated tool 'stock_price_lookup_tool'
2026-08-25 23:01:50,588 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-25 23:01:50,607 - INFO - Tool Call: get_stock_price for ticker 'AAPL'


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The simulated stock price for AAPL is $178.15.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 86344487-8bf8-4652-bd15-305077e9da35                                                                       │
│  Final Output: The simulated stock price for AAPL is $178.15.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


---------------------------------
## Crew execution finished.

Final Result:
 The simulated stock price for AAPL is $178.15.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## CrewAI + Ollama

In [5]:
# Same tool-use example, running on a local model served by Ollama.
# Prerequisites: install Ollama from https://ollama.com, then pull the model:
#   ollama pull qwen2.5:7b

import os
from crewai import Agent, Task, Crew, LLM
from crewai.tools import tool
import logging

# --- Best Practice: Configure Logging ---
# A basic logging setup helps in debugging and tracking the crew's execution.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# --- Point CrewAI at the local Ollama server ---
# CrewAI routes models through LiteLLM, so the model name carries an 'ollama/' prefix.
# Ollama runs locally, so no API key is needed.
# Tool calling needs a model that is good at it; qwen2.5:7b handles this reliably.
OLLAMA_MODEL = "ollama/qwen2.5:7b"  # any model from `ollama list`, e.g. ollama/llama3.1:8b

ollama_llm = LLM(model=OLLAMA_MODEL, base_url="http://localhost:11434")


# --- 1. Refactored Tool: Returns Clean Data ---
# The tool now returns raw data (a float) or raises a standard Python error.
# This makes it more reusable and forces the agent to handle outcomes properly.
@tool("Stock Price Lookup Tool")
def get_stock_price(ticker: str) -> float:
    """
    Fetches the latest simulated stock price for a given stock ticker symbol.
    Returns the price as a float. Raises a ValueError if the ticker is not found.
    """
    logging.info(f"Tool Call: get_stock_price for ticker '{ticker}'")
    simulated_prices = {
        "AAPL": 178.15,
        "GOOGL": 1750.30,
        "MSFT": 425.50,
    }
    price = simulated_prices.get(ticker.upper())

    if price is not None:
        return price
    else:
        # Raising a specific error is better than returning a string.
        # The agent is equipped to handle exceptions and can decide on the next action.
        raise ValueError(f"Simulated price for ticker '{ticker.upper()}' not found.")


# --- 2. Define the Agent ---
# The agent definition remains the same, but it will now leverage the improved tool.
financial_analyst_agent = Agent(
  role='Senior Financial Analyst',
  goal='Analyze stock data using provided tools and report key prices.',
  backstory="You are an experienced financial analyst adept at using data sources to find stock information. You provide clear, direct answers.",
  verbose=True,
  tools=[get_stock_price],
  # Allowing delegation can be useful, but is not necessary for this simple task.
  allow_delegation=False,
  llm=ollama_llm, # Run this agent on the local model.
)

# --- 3. Refined Task: Clearer Instructions and Error Handling ---
# The task description is more specific and guides the agent on how to react
# to both successful data retrieval and potential errors.
analyze_aapl_task = Task(
  description=(
      "What is the current simulated stock price for Apple (ticker: AAPL)? "
      "Use the 'Stock Price Lookup Tool' to find it. "
      "If the ticker is not found, you must report that you were unable to retrieve the price."
  ),
  expected_output=(
      "A single, clear sentence stating the simulated stock price for AAPL. "
      "For example: 'The simulated stock price for AAPL is $178.15.' "
      "If the price cannot be found, state that clearly."
  ),
  agent=financial_analyst_agent,
)

# --- 4. Formulate the Crew ---
# The crew orchestrates how the agent and task work together.
financial_crew = Crew(
  agents=[financial_analyst_agent],
  tasks=[analyze_aapl_task],
  verbose=True # Set to False for less detailed logs in production
)

# --- 5. Run the Crew ---
async def main():
    """Main function to run the crew."""
    # No API key check here: Ollama serves the model locally.
    print("\n## Starting the Financial Crew...")
    print("---------------------------------")

    # The notebook already runs an event loop, so use the async kickoff.
    # The synchronous kickoff() raises RuntimeError when called from one.
    result = await financial_crew.kickoff_async()

    print("\n---------------------------------")
    print("## Crew execution finished.")
    print("\nFinal Result:\n", result)

await main()


## Starting the Financial Crew...
---------------------------------


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 44314983-6db4-40e1-8757-4804f66938b8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│  ID: 976e6c6b-1a61-4699-b926-4d921ae8187d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│  Task: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-08-25 23:01:51,395 - INFO - OpenAI: Successfully validated tool 'stock_price_lookup_tool'
2026-08-25 23:01:53,392 - INFO - HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
2026-08-25 23:01:53,398 - INFO - Tool Call: get_stock_price for ticker 'AAPL'


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The simulated stock price for AAPL is $178.15.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: What is the current simulated stock price for Apple (ticker: AAPL)? Use the 'Stock Price Lookup Tool'    │
│  to find it. If the ticker is not found, you must report that you were unable to retrieve the price.            │
│  Agent: Senior Financial Analyst                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 44314983-6db4-40e1-8757-4804f66938b8                                                                       │
│  Final Output: The simulated stock price for AAPL is $178.15.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


---------------------------------
## Crew execution finished.

Final Result:
 The simulated stock price for AAPL is $178.15.


# Google ADK

## Google Search

In [6]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types

# Define variables required for Session setup and Agent execution
APP_NAME="Google Search_agent"
USER_ID="user1234"
SESSION_ID="1234"


# Define Agent with access to search tool
root_agent = Agent(
   name="basic_search_agent",
   model="gemini-flash-latest",
   description="Agent to answer questions using Google Search.",
   instruction="I can answer your questions by searching the internet. Just ask me anything!",
   tools=[google_search] # Google Search is a pre-built tool to perform Google searches.
)

# Agent Interaction
async def call_agent(query):
   """
   Helper function to call the agent with a query.
   """

   # Session and Runner
   session_service = InMemorySessionService()
   session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
   runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

   content = types.Content(role='user', parts=[types.Part(text=query)])
   events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)


   async for event in events:
       if event.is_final_response():
           final_response = event.content.parts[0].text
           print("Agent Response: ", final_response)

# The notebook already runs an event loop, so await the coroutine directly.
await call_agent("what's the latest ai news?")

2026-08-25 23:01:54,435 - INFO - Sending out request, model: gemini-flash-latest, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-08-25 23:01:54,436 - INFO - AFC is enabled with max remote calls: 10.
2026-08-25 23:01:54,436 - WARNING - Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.
2026-08-25 23:02:01,056 - INFO - Response received from the model.


Agent Response:  ### **1. Frontier Models & The Open-Source Wave**
* **Massive Open-Weight Releases:** Alibaba released **Qwen3.8-Max** (a 2.4-trillion-parameter model with ~95B active parameters and 1M context) as well as its dense **Qwen3.8-27B** model under Apache 2.0, narrowing the gap between open-source weights and top proprietary frontier systems.
* **Rapid Model Iterations:** Frontier labs continue rapid release cycles across flagship lines—including Anthropic's **Claude Opus 5**, Google's fast-turnaround **Gemini** releases, and OpenAI's **GPT-5.6** tiers—with benchmarks showing increasing specialization (e.g., coding agents, multi-step planning, long-context document analysis).

---

### **2. Shift to Autonomous Agents & Digital Coworkers**
* **ChatGPT Work & Agentic Tools:** OpenAI introduced **ChatGPT Work**, an agent designed to automate complex, multi-step workflows across desktop and web environments (producing slides, data sheets, and applications autonomously).
* **Age

## Calculator

In [7]:
import os, getpass
from typing import List
from dotenv import load_dotenv
import logging
from google.adk.agents import Agent as ADKAgent, LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.adk.code_executors import BuiltInCodeExecutor
from google.genai import types

# Define variables required for Session setup and Agent execution
APP_NAME="calculator"
USER_ID="user1234"
SESSION_ID="session_code_exec_async"


# Agent Definition
code_agent = LlmAgent(
   name="calculator_agent",
   model="gemini-flash-latest",
   code_executor=BuiltInCodeExecutor(),
   instruction="""You are a calculator agent.
   When given a mathematical expression, write and execute Python code to calculate the result.
   Return only the final numerical result as plain text, without markdown or code blocks.
   """,
   description="Executes Python code to perform calculations.",
)

# Agent Interaction (Async)
async def call_agent_async(query):

   # Session and Runner
   session_service = InMemorySessionService()
   session = await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
   runner = Runner(agent=code_agent, app_name=APP_NAME, session_service=session_service)

   content = types.Content(role='user', parts=[types.Part(text=query)])
   print(f"\n--- Running Query: {query} ---")
   final_response_text = "No final text response captured."
   try:
       # Use run_async
       async for event in runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content):
           print(f"Event ID: {event.id}, Author: {event.author}")

           # --- Check for specific parts FIRST ---
           # has_specific_part = False
           if event.content and event.content.parts and event.is_final_response():
               for part in event.content.parts: # Iterate through all parts
                   if part.executable_code:
                       # Access the actual code string via .code
                       print(f"  Debug: Agent generated code:\n```python\n{part.executable_code.code}\n```")
                       has_specific_part = True
                   elif part.code_execution_result:
                       # Access outcome and output correctly
                       print(f"  Debug: Code Execution Result: {part.code_execution_result.outcome} - Output:\n{part.code_execution_result.output}")
                       has_specific_part = True
                   # Also print any text parts found in any event for debugging
                   elif part.text and not part.text.isspace():
                       print(f"  Text: '{part.text.strip()}'")
                       # Do not set has_specific_part=True here, as we want the final response logic below

               # --- Check for final response AFTER specific parts ---
               text_parts = [part.text for part in event.content.parts if part.text]
               final_result = "".join(text_parts)
               print(f"==> Final Agent Response: {final_result}")

   except Exception as e:
       print(f"ERROR during agent run: {e}")
   print("-" * 30)

# Main async function to run the examples
async def main():
   await call_agent_async("Calculate the value of (5 + 7) * 3")
   await call_agent_async("What is 10 factorial?")


# Execute the main async function.
# The notebook already runs an event loop, so await the coroutine directly.
await main()

2026-08-25 23:02:16,408 - INFO - Sending out request, model: gemini-flash-latest, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-08-25 23:02:16,408 - INFO - AFC is enabled with max remote calls: 10.



--- Running Query: Calculate the value of (5 + 7) * 3 ---


2026-08-25 23:02:17,956 - INFO - Response received from the model.
2026-08-25 23:02:17,994 - INFO - Sending out request, model: gemini-flash-latest, backend: GoogleLLMVariant.GEMINI_API, stream: False
2026-08-25 23:02:17,995 - INFO - AFC is enabled with max remote calls: 10.


Event ID: b767569e-0934-439d-958e-9827379a9ec7, Author: calculator_agent
Event ID: cc7cec84-bc73-4eb8-b22e-d4f2fca0eb97, Author: calculator_agent
  Debug: Agent generated code:
```python
result = (5 + 7) * 3
print(result)

```
  Debug: Code Execution Result: Outcome.OUTCOME_OK - Output:
36

  Text: '36'
==> Final Agent Response: 36
------------------------------

--- Running Query: What is 10 factorial? ---


2026-08-25 23:02:19,576 - INFO - Response received from the model.


Event ID: 302a32bf-fd4b-4830-9ef5-7962b1ba3632, Author: calculator_agent
Event ID: 64e213c7-4fb8-456c-bab6-5d81e0539c8d, Author: calculator_agent
  Debug: Agent generated code:
```python
import math
print(math.factorial(10))
```
  Debug: Code Execution Result: Outcome.OUTCOME_OK - Output:
3628800

  Text: '3628800'
==> Final Agent Response: 3628800
------------------------------


## Vertex AI Search

In [25]:
import os

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import VertexAiSearchTool
from google.genai import types

# --- Configuration ---
# Vertex AI Search is a Google Cloud service, so this example does NOT run on a
# GOOGLE_API_KEY from AI Studio. Calling it in Developer API mode fails with
# "retrieval parameter is only supported in Gemini Enterprise Agent Platform mode".
#
# To run it you need, in .env or the environment:
#   GOOGLE_GENAI_USE_VERTEXAI=TRUE
#   GOOGLE_CLOUD_PROJECT=<your Google Cloud project id>
#   GOOGLE_CLOUD_LOCATION=<region, e.g. us-central1>
#   DATASTORE_ID=projects/<project>/locations/<location>/collections/default_collection/dataStores/<datastore>
# plus credentials: gcloud auth application-default login
#
# The DATASTORE_ID points at a Vertex AI Search data store that you created and
# filled with your own documents; the queries below assume it holds the Q2
# strategy documents.

DATASTORE_ID = os.environ.get("DATASTORE_ID")
MODEL = "gemini-flash-latest"

# --- Application Constants ---
APP_NAME = "vsearch_app"
USER_ID = "user_123"  # Example User ID
SESSION_ID = "session_456"  # Example Session ID


def missing_requirements() -> list[str]:
    """Returns the settings this example needs but cannot find in the environment."""
    missing = []
    if not DATASTORE_ID:
        missing.append("DATASTORE_ID")
    if os.environ.get("GOOGLE_GENAI_USE_VERTEXAI", "").upper() not in ("1", "TRUE"):
        missing.append("GOOGLE_GENAI_USE_VERTEXAI=TRUE")
    if not os.environ.get("GOOGLE_CLOUD_PROJECT"):
        missing.append("GOOGLE_CLOUD_PROJECT")
    if not os.environ.get("GOOGLE_CLOUD_LOCATION"):
        missing.append("GOOGLE_CLOUD_LOCATION")
    return missing


# --- Agent Definition ---
# ADK has no VSearchAgent class: grounding on a data store is a tool that any
# agent can carry, so this is a normal Agent holding a VertexAiSearchTool.
def build_vsearch_agent() -> Agent:
    """Creates an agent grounded on the configured Vertex AI Search data store."""
    return Agent(
        name="q2_strategy_vsearch_agent",
        description="Answers questions about Q2 strategy documents using Vertex AI Search.",
        model=MODEL,
        instruction=(
            "Answer questions using the Vertex AI Search tool to find information "
            "in the internal documents. Always cite sources when available."
        ),
        tools=[VertexAiSearchTool(data_store_id=DATASTORE_ID)],
        generate_content_config=types.GenerateContentConfig(temperature=0.0),
    )


# --- Agent Invocation Logic ---
async def call_vsearch_agent_async(runner: Runner, query: str):
    """Sends one query to the agent and prints the answer with its sources."""
    print(f"User: {query}")
    print("Agent: ", end="", flush=True)

    try:
        content = types.Content(role="user", parts=[types.Part(text=query)])

        async for event in runner.run_async(
            user_id=USER_ID,
            session_id=SESSION_ID,
            new_message=content,
        ):
            if event.is_final_response() and event.content and event.content.parts:
                print("".join(part.text for part in event.content.parts if part.text))

                # Grounding chunks are the documents the answer was based on.
                chunks = (
                    event.grounding_metadata.grounding_chunks
                    if event.grounding_metadata
                    else None
                )
                if chunks:
                    print(f"  (Source Attributions: {len(chunks)} sources found)")
                else:
                    print("  (No grounding metadata found)")
                print("-" * 30)

    except Exception as e:
        print(f"\nAn error occurred: {e}")
        print("Check that DATASTORE_ID is a full resource name and that your")
        print("credentials have access to that data store.")
        print("-" * 30)


# --- Execution ---
async def main():
    missing = missing_requirements()
    if missing:
        print("Skipping execution. Missing configuration:")
        for item in missing:
            print(f"  - {item}")
        print("See the comments at the top of this cell for how to set these.")
        return

    session_service = InMemorySessionService()
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
    runner = Runner(
        agent=build_vsearch_agent(),
        app_name=APP_NAME,
        session_service=session_service,
    )

    # Replace with questions relevant to YOUR datastore content.
    await call_vsearch_agent_async(runner, "Summarize the main points about the Q2 strategy document.")
    await call_vsearch_agent_async(runner, "What safety procedures are mentioned for lab X?")


# The notebook already runs an event loop, so await the coroutine directly.
await main()


Skipping execution. Missing configuration:
  - DATASTORE_ID
  - GOOGLE_GENAI_USE_VERTEXAI=TRUE
  - GOOGLE_CLOUD_PROJECT
  - GOOGLE_CLOUD_LOCATION
See the comments at the top of this cell for how to set these.
